In [14]:
import sqlite3
import pandas as pd
import ast

con=sqlite3.connect(r"C:\Users\muham\OneDrive\Desktop\ipl\data\raw\ipl.db")
def q(sql):
    return pd.read_sql(sql,con)

In [15]:
def run_sql_file(path):
    with open((path), 'r') as file:
        script = file.read()
    con.executescript(script)
    con.commit()
    print('ran', path)  

In [16]:
run_sql_file(r"C:\Users\muham\OneDrive\Desktop\ipl\sql\test.sql")

ran C:\Users\muham\OneDrive\Desktop\ipl\sql\test.sql


In [17]:
run_sql_file(r"C:\Users\muham\OneDrive\Desktop\ipl\sql\01_nulls_and_blanks.sql")

ran C:\Users\muham\OneDrive\Desktop\ipl\sql\01_nulls_and_blanks.sql


In [18]:
q("""select count(*) as blank_before
from players
where trim(field_pos)='';""")

,blank_before
0,612


In [19]:
q("""select count(*) as null_after
from v_players_clean
where field_pos_clean is null;""")

,null_after
0,700


In [20]:
run_sql_file(r"C:\Users\muham\OneDrive\Desktop\ipl\sql\02_merge_categories.sql")

ran C:\Users\muham\OneDrive\Desktop\ipl\sql\02_merge_categories.sql


In [21]:
q("""select distinct team_name
from teams 
order by team_name;""")  

,team_name
0,Chennai Super Kings
1,Deccan Chargers
2,Delhi Capitals
3,Gujarat Lions
4,Gujarat Titans
5,Kochi Tuskers Kerala
6,Kolkata Knight Riders
7,Lucknow Super Giants
8,Mumbai Indians
9,Pune Warriors


In [22]:
q(""" select distinct team_name_clean
 from v_teams_clean
 order by team_name_clean;""")

,team_name_clean
0,Chennai Super Kings
1,Deccan Chargers
2,Delhi Capitals
3,Gujarat Lions
4,Gujarat Titans
5,Kochi Tuskers Kerala
6,Kolkata Knight Riders
7,Lucknow Super Giants
8,Mumbai Indians
9,Pune Warriors


In [23]:
run_sql_file(r"C:\Users\muham\OneDrive\Desktop\ipl\sql\03_venue_names.sql")

ran C:\Users\muham\OneDrive\Desktop\ipl\sql\03_venue_names.sql


In [24]:
run_sql_file(r"C:\Users\muham\OneDrive\Desktop\ipl\sql\04_dedupe_venues.sql")

ran C:\Users\muham\OneDrive\Desktop\ipl\sql\04_dedupe_venues.sql


In [25]:
q("""select venue_clean, count(*) as matches
from v_matches_venue
group by venue_clean
order by matches desc limit 10;""")

,venue_clean,matches
0,Wankhede Stadium,130
1,Eden Gardens,104
2,MA Chidambaram Stadium,95
3,M Chinnaswamy Stadium,89
4,Rajiv Gandhi International Stadium,87
5,Sawai Mansingh Stadium,66
6,Feroz Shah Kotla,60
7,Dubai International Cricket Stadium,46
8,Arun Jaitley Stadium,41
9,Narendra Modi Stadium,37


In [26]:
q("""select count(*) as rows,count(distinct venue) as names
from venues;""")

,rows,names
0,63,59


In [27]:
q("""select count(*) as venue,city
from venues
group by venue
order by count(*) desc limit 10;""")

,venue,city
0,2,NaN
1,2,Mohali
2,2,NaN
3,2,Mumbai
4,1,Abu Dhabi
5,1,Mumbai
6,1,Mumbai
7,1,Nagpur
8,1,Centurion
9,1,Pune


In [28]:
run_sql_file(r"C:\Users\muham\OneDrive\Desktop\ipl\sql\05_city_fallback.sql")

ran C:\Users\muham\OneDrive\Desktop\ipl\sql\05_city_fallback.sql


In [29]:
run_sql_file(r"C:\Users\muham\OneDrive\Desktop\ipl\sql\06_season_year.sql")

ran C:\Users\muham\OneDrive\Desktop\ipl\sql\06_season_year.sql


In [30]:
q(""" select season,
         count(*) as matches
from matches
where season ='2020/21'
group by season;""")

,season,matches
0,2020/21,60


In [31]:
q(""" select distinct season, season_year
from v_season
where season ='2020/21';""")

,season,season_year
0,2020/21,2020


In [32]:
run_sql_file(r"C:\Users\muham\OneDrive\Desktop\ipl\sql\07_win_definition.sql")

ran C:\Users\muham\OneDrive\Desktop\ipl\sql\07_win_definition.sql


In [33]:
run_sql_file(r"C:\Users\muham\OneDrive\Desktop\ipl\sql\08_matches_clean.sql")

ran C:\Users\muham\OneDrive\Desktop\ipl\sql\08_matches_clean.sql


In [34]:
q("""SELECT * FROM matches_clean""")

,match_id,venue_clean,city_clean,season_year,result,match_winner,player_of_match,toss_decision
0,335982,M Chinnaswamy Stadium,Bangalore,2008,win,Kolkata Knight Riders,BB McCullum,field
1,1082591,Rajiv Gandhi International Stadium,Hyderabad,2017,win,Sunrisers Hyderabad,Yuvraj Singh,field
2,1082592,Maharashtra Cricket Association Stadium,Pune,2017,win,Rising Pune Supergiant,SPD Smith,field
3,1082593,Saurashtra Cricket Association Stadium,Rajkot,2017,win,Kolkata Knight Riders,CA Lynn,field
4,1082594,Holkar Cricket Stadium,Indore,2017,win,Punjab Kings,GJ Maxwell,field
...,...,...,...,...,...,...,...,...
1063,1529279,Sawai Mansingh Stadium,Jaipur,2026,win,Sunrisers Hyderabad,Ishan Kishan,field
1064,1529280,MA Chidambaram Stadium,Chennai,2026,win,Gujarat Titans,K Rabada,field
1065,1529282,Arun Jaitley Stadium,Delhi,2026,win,Royal Challengers Bangalore,JR Hazlewood,field
1066,1529284,Wankhede Stadium,Mumbai,2026,win,Sunrisers Hyderabad,H Klaasen,bat


In [35]:

d= pd.read_sql('select * from deliveries where fielders_involved is not null', con)
d['fielders']= d['fielders_involved'].apply(ast.literal_eval)

d=d.explode('fielders')

d['fielders'].value_counts().head()

fielders
MS Dhoni          258
KD Karthik        228
RV Uthappa        169
AB de Villiers    148
V Kohli           142
Name: count, dtype: int64